# Forte — первый воспроизводимый эксперимент

Это **исследовательский бинарный baseline**, не готовый сервис и не юридическая экспертиза. Обучается линейный классификатор поверх замороженных признаков ResNet18; большая нейросеть с нуля не обучается. Типы нарушений и точные моменты этот baseline не определяет.

Используются только выданные обучающие материалы. Не загружайте закрытые тесты жюри. Никаких внешних AI API. Материалы не публикуются; Google Drive автоматически не подключается.

Запускайте ячейки по порядку. Можно выбрать GPU, но этот небольшой эксперимент работает и на CPU. Результаты сессии нужно скачать до её завершения.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "av>=14,<20", "openpyxl>=3.1,<4", "scikit-learn>=1.5,<2", "tqdm>=4.66,<5"])
import torch, torchvision
print("PyTorch:", torch.__version__, "Torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU; запуск допустим")


## 1. Загрузка компактного архива

Через панель «Файлы» слева загрузите **forte_frames.zip** и дождитесь завершения. Он содержит 24 равномерных кадра на уникальное видео, исходные временные метки, хеши и Excel. PNG сохраняет кадры без потерь после уменьшения с полями до 224×224. Исходные MOV не изменены. Файлы находятся только в сессии Colab.


In [ ]:
from pathlib import Path
ARCHIVE = Path("/content/forte_frames.zip")
assert ARCHIVE.exists(), "Загрузите forte_frames.zip через панель Файлы слева"
assert ARCHIVE.stat().st_size > 0
print("Компактный архив:", round(ARCHIVE.stat().st_size / 1024**2, 1), "MiB")


## 2. Аудит и объединение дубликатов
Исходный Excel не изменяется. Для совпадающих файлов сохраняются все исходные таймкоды. Ошибки меток и непарные файлы останавливают запуск.


In [ ]:
import hashlib, json, re, zipfile, time, platform
from collections import defaultdict
import av, numpy as np, pandas as pd
from openpyxl import load_workbook
BASE = Path("/content/forte_experiment")
DATA = BASE / "data"
OUT = BASE / "results"
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as z:
    if sum(i.file_size for i in z.infolist()) > 2 * 1024**3:
        raise ValueError("Архив превышает лимит распакованного объёма 2 GiB")
    for i in z.infolist():
        target = (DATA / i.filename).resolve()
        if not target.is_relative_to(DATA.resolve()):
            raise ValueError("Небезопасный путь в ZIP")
        if i.is_dir():
            continue
        if target.suffix.lower() not in {".png", ".json", ".xlsx"}:
            raise ValueError("В архиве ожидаются только PNG, JSON и XLSX")
    z.extractall(DATA)

def canon(s):
    return Path(str(s)).stem.strip().upper().replace("Х", "X")

def parse_time(s):
    if s is None or str(s).strip() == "":
        return None
    if isinstance(s, (int, float)):
        return float(s)
    s = str(s).lower().strip()
    m, sec = re.search(r"(\d+)\s*мин", s), re.search(r"(\d+)\s*сек", s)
    if m or sec:
        return float((int(m[1])*60 if m else 0) + (int(sec[1]) if sec else 0))
    return float(s)

books = list(DATA.rglob("open_labels.xlsx"))
assert len(books) == 1, "Нужен ровно один open_labels.xlsx"
wb = load_workbook(books[0], read_only=True, data_only=True)
rows = list(wb["labels"].values)
wb.close()
labels = {}
for name, t, label, *_ in rows[1:]:
    if not name:
        continue
    k = canon(name)
    assert k not in labels, f"Дублирующийся ID в Excel: {k}"
    label = str(label).strip().lower()
    assert label in {"нарушение", "без нарушения"}, f"Неизвестная метка: {label}"
    labels[k] = {"label": int(label == "нарушение"), "time": parse_time(t)}
unique = json.loads((DATA/"frame_manifest.json").read_text(encoding="utf-8"))
assert len({r["sha256"] for r in unique}) == len(unique), "Повторяющиеся исходные видео"
assert {k for r in unique for k in r["source_ids"]} == set(labels), "Состав не совпал с Excel"
for r in unique:
    assert all(labels[k]["label"] == r["label"] for k in r["source_ids"])
    expected_times = sorted({labels[k]["time"] for k in r["source_ids"] if labels[k]["time"] is not None})
    assert r["annotation_times"] == expected_times, "Потеряна исходная отметка"
    assert all(0 <= t < r["duration"] for t in expected_times)
    assert len(r["frames"]) == 24
    for f in r["frames"]:
        p = (DATA / f["path"]).resolve()
        assert p.is_relative_to(DATA.resolve()) and p.suffix.lower() == ".png"
        assert hashlib.sha256(p.read_bytes()).hexdigest() == f["sha256"], "Ошибка целостности кадра"
        assert 0 <= f["timestamp"] < r["duration"]
# Ручные предварительные группы по визуальной постановке; не признаки модели.
# Не являются доказанно независимыми съёмочными сессиями. Уточнить после разметки.
def scene_group(r):
    n = int(r["id"].split("-")[-1])
    for top, group in [(5,"setup_1"),(14,"setup_2"),(19,"setup_3"),(29,"setup_4"),(33,"setup_5"),(39,"setup_6"),(46,"setup_7")]:
        if n <= top:
            return group
    if n in (47,48):
        return "setup_1"
    raise ValueError("Для нового видео укажите группу съёмки вручную")
for r in unique:
    r["group"] = scene_group(r)
(OUT/"manifest.json").write_text(json.dumps(unique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Файлов: {len(labels)}; уникальных: {len(unique)}")
print("Классы уникальных:", pd.Series([r["label"] for r in unique]).value_counts().to_dict())
print("Дубликаты:", [r["source_ids"] for r in unique if len(r["source_ids"]) > 1])
print("Несколько отметок:", [(r["source_ids"],r["annotation_times"]) for r in unique if len(r["annotation_times"]) > 1])
display(pd.DataFrame(unique)[["id","source_ids","label","annotation_times","duration","group"]])


## 3. Признаки видео — без использования меток и таймкодов
24 равномерных кадра, исходные временные метки, полное изображение с полями вместо центральной обрезки. ResNet18 заморожен. Обучается только следующий классификатор. Это быстрый ориентир; маленькие предметы и сложный порядок действий он может не различать.

Веса ResNet18 загружаются из официального источника torchvision один раз. При переносе используются сохранённые локальные веса. Лицензии конкретных весов и данных предобучения нужно проверить перед коммерческой передачей.


In [ ]:
from PIL import Image, ImageOps
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLES = 24
MODEL_VERSION = "resnet18_imagenet1k_v1_lossless_frames224_24_mean_std_delta_v1"
encoder = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
encoder.fc = torch.nn.Identity()
encoder = encoder.eval().to(DEVICE)
for param in encoder.parameters():
    param.requires_grad_(False)
FEATURES = BASE / MODEL_VERSION
FEATURES.mkdir(exist_ok=True)

def frame_at(path, target):
    with av.open(str(path)) as c:
        s = c.streams.video[0]
        c.seek(int(target / s.time_base), stream=s, backward=True)
        last = None
        for f in c.decode(s):
            last = f
            if f.time is not None and f.time >= target:
                return f.to_image(), float(f.time)
        if last is None or last.time is None:
            raise RuntimeError("Не удалось декодировать кадр")
        return last.to_image(), float(last.time)

def preprocess(im):
    im = ImageOps.pad(im.convert("RGB"), (224,224), method=Image.Resampling.BILINEAR, color=(124,116,104))
    return TF.normalize(TF.to_tensor(im), [0.485,0.456,0.406], [0.229,0.224,0.225])

def embed_video(r):
    cache = FEATURES / (r["sha256"] + ".npz")
    if cache.exists():
        with np.load(cache, allow_pickle=False) as d:
            return d["feature"].copy()
    imgs, timestamps = [], []
    for item in r["frames"]:
        with Image.open(DATA / item["path"]) as im:
            imgs.append(preprocess(im))
        timestamps.append(item["timestamp"])
    with torch.inference_mode():
        embeddings = torch.cat([encoder(torch.stack(imgs[i:i+8]).to(DEVICE)).cpu() for i in range(0,len(imgs),8)]).numpy()
    feature = np.concatenate([embeddings.mean(0), embeddings.std(0), np.abs(np.diff(embeddings,axis=0)).mean(0)]).astype(np.float32)
    np.savez_compressed(cache, feature=feature, timestamps=np.asarray(timestamps))
    return feature

started = time.perf_counter()
X = np.stack([embed_video(r) for r in tqdm(unique, desc="Уникальные видео")])
y = np.array([r["label"] for r in unique])
groups = np.array([r["group"] for r in unique])
feature_seconds = time.perf_counter() - started
assert np.isfinite(X).all()
np.savez_compressed(OUT/"features.npz", X=X, y=y, groups=groups, ids=np.array([r["id"] for r in unique]))
print("Матрица:", X.shape, "Время извлечения:", round(feature_seconds,1), "секунд")


## 4. Честная предварительная проверка по группам
Каждая постановка по очереди целиком исключается из обучения. Масштабирование признаков учится только на обучающей части. Дубликаты уже объединены. Порог 0.5 фиксирован заранее, настройки не подбираются по проверочным ответам.

Это **не закрытый тест**, группы предварительные, независимого нового ракурса нет. Низкое качество — полезный результат: оно покажет, что одного распознавания обстановки недостаточно. Не использовать оценку на обучающих видео как качество продукта.


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score, confusion_matrix

def make_model():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.01, class_weight="balanced", max_iter=3000, random_state=42))
oof = np.full(len(y), np.nan)
folds = []
for train, valid in LeaveOneGroupOut().split(X,y,groups):
    assert not (set(groups[train]) & set(groups[valid]))
    assert len(np.unique(y[train])) == 2
    clf = make_model().fit(X[train], y[train])
    oof[valid] = clf.predict_proba(X[valid])[:,1]
    folds.append({"held_out_group":str(groups[valid][0]), "train_ids":[unique[i]["id"] for i in train], "valid_ids":[unique[i]["id"] for i in valid]})
assert np.isfinite(oof).all()
pred = (oof >= .5).astype(int)
metrics = {"evaluation":"leave-one-provisional-scene-group-out; not an independent hidden test",
           "n_unique_videos":len(y), "threshold":0.5,
           "precision":float(precision_score(y,pred,zero_division=0)),
           "recall":float(recall_score(y,pred,zero_division=0)),
           "f1":float(f1_score(y,pred,zero_division=0)),
           "balanced_accuracy":float(balanced_accuracy_score(y,pred)),
           "confusion_matrix_true_rows_pred_columns":confusion_matrix(y,pred,labels=[0,1]).tolist(),
           "feature_extraction_sec":feature_seconds}
(OUT/"metrics.json").write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding="utf-8")
(OUT/"folds.json").write_text(json.dumps(folds,ensure_ascii=False,indent=2),encoding="utf-8")
predictions = pd.DataFrame({"id":[r["id"] for r in unique],"group":groups,"actual":y,"prediction":pred,"score_not_calibrated":oof,"error":pred!=y})
predictions.to_csv(OUT/"predictions.csv",index=False)
print(json.dumps(metrics, ensure_ascii=False, indent=2))
display(predictions.sort_values("error",ascending=False))


## 5. Обучение на полном открытом наборе и перенос
Эта модель обучена на всех уникальных открытых видео только ПОСЛЕ получения out-of-fold оценок. Оценки выше относятся к отдельным моделям проверочных разбиений, не к оценке этой модели на её обучении.

Экспорт содержит локальные веса, коэффициенты, конфигурацию, версии и отчёт. Он не содержит исходных видео. Формат классификатора — NPZ без pickle.


In [ ]:
import importlib.metadata as metadata
final_model = make_model().fit(X,y)
scaler, linear = final_model.steps[0][1], final_model.steps[1][1]
np.savez_compressed(OUT/"classifier.npz", mean=scaler.mean_, scale=scaler.scale_, coef=linear.coef_, intercept=linear.intercept_)
torch.save({k:v.cpu() for k,v in encoder.state_dict().items()}, OUT/"encoder.pth")
config = {"model_version":MODEL_VERSION,"samples":SAMPLES,"threshold":0.5,"frame_size":224,
          "labels":{"0":"no_violation","1":"violation"},
          "limitations":["Research binary baseline only","No event type or reliable timestamp","No legal conclusions","Scores are not calibrated probabilities","Not validated on an independent camera angle"]}
(OUT/"config.json").write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding="utf-8")
versions = {p:metadata.version(p) for p in ["torch","torchvision","numpy","pandas","Pillow","av","openpyxl","scikit-learn"]}
(OUT/"versions.json").write_text(json.dumps(versions,indent=2),encoding="utf-8")
(OUT/"environment.txt").write_text(platform.platform()+"\n"+sys.version+"\n"+"\n".join(f"{k}=={v}" for k,v in versions.items()),encoding="utf-8")
# Следующий этап: ручная разметка. Не заполняем догадками типы и координаты.
annotation_rows = [{"video_id":r["id"],"source_times":json.dumps(r["annotation_times"]),"event_type":"","start_sec":"","end_sec":"","observable_fact":"","rule_source":"","reviewed":False} for r in unique if r["label"]]
pd.DataFrame(annotation_rows).to_csv(OUT/"annotation_template.csv",index=False)
(OUT/"README.txt").write_text("Forte research baseline. Not a completed competition service.\nLoad encoder with resnet18(weights=None), replace fc with Identity, then torch.load(..., weights_only=True).\nPreprocess: RGB, PIL ImageOps.pad 224x224 BILINEAR fill (124,116,104), tensor and ImageNet normalization.\nSample 24 uniform timestamps [0,duration-0.1], use PTS, concatenate embedding mean/std/mean absolute adjacent difference.\nScore: sigmoid(((feature-mean)/scale) @ coef.T + intercept); threshold 0.5.\nDo not infer event time, type or legal rule from this classifier.\nReview licenses of all artifacts before commercial delivery.\n",encoding="utf-8")
# Проверка: экспортированные коэффициенты повторяют обученный классификатор.
with np.load(OUT/"classifier.npz",allow_pickle=False) as exported:
    z = ((X-exported["mean"])/exported["scale"]) @ exported["coef"].T + exported["intercept"]
    probabilities = 1/(1+np.exp(-np.clip(z[:,0],-700,700)))
assert np.allclose(probabilities, final_model.predict_proba(X)[:,1],atol=1e-6)
BUNDLE = BASE / "forte_baseline_bundle.zip"
with zipfile.ZipFile(BUNDLE,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file():
            z.write(p,p.name)
print("Экспорт проверен:", BUNDLE, round(BUNDLE.stat().st_size/1024**2,1),"MiB")
print("Важно: скачайте архив. Файлы сессии Colab могут исчезнуть после её завершения.")


In [ ]:
from google.colab import files
# Компактная резервная копия: обученный классификатор и отчёт.
# encoder.pth остаётся в полном forte_baseline_bundle.zip; без него это не автономная модель.
CORE_BUNDLE = BASE / "forte_baseline_core.zip"
with zipfile.ZipFile(CORE_BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p.name != "encoder.pth":
            z.write(p, p.name)
print("Компактная резервная копия:", CORE_BUNDLE, CORE_BUNDLE.stat().st_size, "bytes")
files.download(str(CORE_BUNDLE))
# Полный архив скачайте через панель Файлы: forte_experiment/forte_baseline_bundle.zip


## Что дальше
1. Разобрать ложные тревоги и пропуски из predictions.csv.
2. Уточнить события и публичные нормативные основания в annotation_template.csv.
3. Разметить объекты/состояния и дообучить детектор, добавить временные правила.
4. Проверить локальный запуск на RTX 4050 6 GB, другой ракурс и лимит 5 минут.
5. Добавить интерфейс и обязательный API. Этот блокнот их не заменяет.

Ссылки: [ResNet18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html), [Colab FAQ](https://research.google.com/colaboratory/faq.html).
